In [1]:
from IPython.display import HTML
with open("custom/style.css", "r") as f:
    custom_style = f.read()
hide_cell_style = "<style>.jp-Cell:has(.hide-me) { display: yes; }</style>"
HTML(f"<div class='hide-me'></div><style>{custom_style}</style>{hide_cell_style}")

<h1><center><font color='blue'>Introduction to Parallel Programming with MPI<br><br>Basics</font></center></h1>

# <font color='blue'>Basics</font>

## The message passing paradigm

<img src="figs/comm-network.svg" alt="comm-network.svg" style="float: right; margin-right: 15px;" width="40%">
<!--```{image} figs/comm-network.svg
:width: 500px
```-->

- Distributed-memory architecture
  - Each process(or) can only access its <font color='green'>dedicated address space</font>
  - No global shared address space
  - <font color='green'>Data exchange</font> and communication between processes is done
    by <font color='green'>explicitly passing messages</font> through a communication network <p> </p>

- Message passing library
  - Should be flexible, efficient and portable
  - Hide communication hardware and software layers from application developer
- Widely accepted standard in HPC / numerical simulation: <font color='green'>Message Passing Interface (MPI)</font>
- Process-based approach: <font color='green'>All variables are local!</font>
- Same program on each processor/machine (SPMD)
- The program is written in a <font color='green'>sequential language</font> (Fortran/C[++]): but not restricted only to these two programming languages
- <font color='green'>Data exchange</font> between processes: send/receive messages via <font color='green'>MPI library calls</font>
  - <font color='red'>No automatic workload distribution</font>

## The MPI Standard

<img src="figs/MPI-standard.svg" alt="MPI-standard.svg" style="float: right; margin-right: 15px;" width="25%">
<!--```{image} figs/MPI-standard.svg
:width: 300px
```-->

- **<font color='green'>MPI forum:</font>** defines MPI standard / library subroutine interfaces
- Latest standard in use: MPI 4.1 (November 2, 2023), MPI 5.0 was approved by the MPI Forum on June 5, 2025
- Members (approx. 60) of MPI standard forum
  - Application developers
  - Research institutes & computing centers
  - Manufacturers of supercomputers & software designers
- Successful implementations
  - open-source
    - <font color='blue'>[MPICH](https://www.mpich.org/)</font>
    - <font color='blue'>[mvapich](https://mvapich.cse.ohio-state.edu/)</font>
    - <font color='blue'>[OpenMPI](https://www.open-mpi.org/)</font>
  - vendor libraries
    - <font color='blue'>Intel</font>
    - <font color='blue'>Cray</font>
- The official version of the MPI standard documents:<br> <font color='blue'><http://www.mpi-forum.org/></font>

## MPI goals and scope

<img src="figs/MPI-hardware-software-layers.svg" alt="MPI-hardware-software-layers.svg" style="float: right; margin-right: 15px;" width="30%">
<!--```{image} figs/MPI-hardware-software-layers.svg
:width: 300px
```-->

- **<font color='green'>Portability</font>** is main goal: architecture- and hardware-independent code
- <font color='green'>Fortran</font> and <font color='green'>C interfaces</font>
  - <font color='red'>(C++ deprecated)</font>
- Features for supporting <font color='green'>parallel libraries</font>
- Support for <font color='green'>heterogeneous environments</font> (e.g., clusters with compute nodes of different architectures)

## Parallel execution in MPI

<img src="figs/parallel-execution.svg" alt="parallel-execution.svg" style="float: right; margin-right: 15px;" width="30%">
<!--```{image} figs/parallel-execution.svg
:width: 300px
```-->

- Processes run throughout program execution
- MPI startup mechanism:
  - launches tasks/processes, think of executing multiple copies of a program
  - establishes communication context (<font color='green'>*communicator*</font> )
- <font color='blue'>MPI Point-to-point communication:</font>
  - between <font color='blue'>pairs of tasks/processes</font>
- <font color='purple'>MPI Collective communication:</font>
  - between <font color='purple'>all processes</font> or a subgroup
  - barrier, reductions, scatter/gather
- Clean <font color='green'>shutdown</font> by <font color='green'>MPI</font>

## World communicator and rank

<img src="figs/world-communicator.svg" alt="world-communicator.svg" style="float: right; margin-right: 15px;" width="40%">
<!--```{image} figs/world-communicator.svg
:width: 400px
```-->

- Entities must be in a group/community to be able to communicate
- <font color='green'>Communicator</font> is a handle
- <font color='green'>MPI_Init():</font>
  - <font color='green'>MPI_COMM_WORLD</font>
  - all processes
- MPI_COMM_WORLD
  - Fortran and C

## Initialization and finalization

- <font color='green'>Startup command</font> of an MPI application is implementation dependent
- <font color='green'>First call</font> in an MPI program: initialization of parallel machine <br>
```cpp
int MPI_Init(int *argc, char ***argv);
```
- <font color='green'>Last call</font>: clean shutdown of parallel machine <br>
```cpp
int MPI_Finalize();
```
  - Only *<font color='purple'>master</font>* process is guaranteed to continue after finalize
- **Stdout/stderr** of each MPI process
  - usually redirected to console where program was started
  - many options possible, <font color='green'>depending on implementation</font>

## Communicator and rank

- Communicator defines a set of processes (<font color='green'>MPI_COMM_WORLD</font>: all processes)
- <font color='green'>rank:</font> an integer identifying each process within a communicator
  - Obtain rank:
    ```cpp
    int rank;
    int MPI_Comm_rank(MPI_COMM_WORLD, &rank);
    ```
  - rank = 0,1,2,..., (number of processes in communicator - 1)
  - <font color='green'>Not unique:</font> one process may have distinct ranks in <font color='green'>different communicators</font>
  - Obtain number of processes in communicator:
    ```cpp
    int size;
    MPI_Comm_size(MPI_COMM_WORLD, &size);
    ```

## MPI "Hello World!" in C


<div class="callout note">
<strong>Note:</strong>
<span style="color:darkblue">
Never forget that <code>argc</code> and <code>argv</code> are pointers to the original variables!</span>
</div>

```cpp
#include <mpi.h>

int main(char argc, char **argv) {
  int rank, size;

    MPI_Init(&argc, &argv);
    MPI_Comm_size(MPI_COMM_WORLD, &size);
    MPI_Comm_rank(MPI_COMM_WORLD, &rank);

    printf("Hello World! I am %d of %d\n", rank, size);

    MPI_Finalize();
}
```

<div class="callout note">
<strong>Note:</strong>
<span style="color:darkblue">
Communicator required for (almost) all MPI calls</span>
</div>

## MPI "Hello World!" in Fortran

<div class="callout note">
<strong>Note:</strong>
By default, Fortran arguments are passed by reference!
</div>

```fortran
program hello
  use mpi
  implicit none
  integer:: rank, size, ierr
  !include "mpif.h"

  call MPI_INIT(ierr)
  call MPI_COMM_SIZE(MPI_COMM_WORLD,size,ierr)
  call MPI_COMM_RANK(MPI_COMM_WORLD,rank,ierr)
  
  write(*,'(2(a,i))') "Hello World! I am ",rank," of ",size

  call MPI_FINALIZE(ierr)
end program hello
```

<div class="callout note">
<strong>Note:</strong>
Communicator required for (almost) all MPI calls
</div>

## Compiling and running the code

- Compiling/linking
  - <font color='green'>Headers and libs</font> must be found by compiler
  - Most implementations provide wrapper scripts, e.g.,
    - <font color='green'>mpif77 / mpif90</font>
    - <font color='green'>mpicc / mpiCC</font>
  - Behave like normal compilers/linkers
- Running
  - Details are implementation specific
  - Startup wrappers: <font color='green'>mpirun, mpiexec, aprun, poe</font>
    - Job scheduler wrappers: <font color='green'>srun</font>

**Method 1:**
```cpp
$ mpiCC -o hello hello.cc
$ mpirun -np 3 ./hello
Hello World! I am 2 of 3
Hello World! I am 1 of 3
Hello World! I am 0 of 3
```

**Method 2:**
```cpp
$ mpirun -np 1 ./hello : -np 1 ./hello : -np 1 ./hello
Hello World! I am 1 of 3
Hello World! I am 0 of 3
Hello World! I am 2 of 3
```

## Point-to-Point Communication

<div style="text-align: center; font-size: 24px; max-width: 2000px;">
    <span>
        It is a communication between <font color='green'>two processes</font> where a sender (source process) sends message to a receiver (destination process).
    </span>
</div>

- Procedure (C/C++ binding, Fortran binding, Fortran 2008 binding)
- Message data
  - Buffer (<font color='green'>address</font>)
  - Datatype (basic or derived?)
  - Count (number of elements, <font color='green'>not bytes</font>)
- Message envelope
  - Source
  - Destination
  - Tag

<div class="callout note">
<strong>Note:</strong>
<span style="color:darkblue">
You may see C/C++ binding(s) throught the course, they are indeed C bindings which are expected to be used by C++ programmers as well.</span>
</div>

## Point-to-point communication: message envelope

<img src="figs/comm-network.svg" alt="comm-network.svg" style="float: right; margin-right: 15px;" width="45%">
<!--```{image} figs/comm-network.svg
:width: 500px
```-->

- <font color='green'>Which process is sending the message?</font>
- <font color='green'>Where is the data on the sending process?</font>
- <font color='green'>What kind of data is being sent?</font>
- <font color='green'>How much data is there?</font><p></p>
- <font color='blue'>Which process is receiving the message?</font>
- <font color='blue'>Where should the data be left on the receiving process?</font>
- <font color='blue'>How much data is the receiving process prepared to accept?</font><p></p>
- <font color='green'>Sender</font> and <font color='blue'>receiver</font> must pass their information to MPI separately

## MPI point-to-point communication

<img src="figs/Point-to-Point-Anim.gif" alt="Point-to-Point-Anim.gif"  width="60%">
<!--```{image} figs/Point-to-Point-Anim.gif
:width: 800px
```-->

- Processes communicate by sending and receiving messages
- MPI message: <font color='green'>array</font> of elements of a <font color='green'>particular type</font>

- Data types
  - <font color='green'>Basic</font>
  - <font color='green'>MPI derived types</font>

## MPI_SEND

- C/C++ binding:
```cpp
#include <mpi.h>
int MPI_Send(const void *buf, int count, MPI_Datatype datatype, int dest,int tag, MPI_Comm comm)
```
- <font color='green'>buf:</font> address of the first entry of the buffer to be sent
- <font color='green'>count:</font> number of elements to be sent (<font color='red'>note that it is not bytes!</font>)
- <font color='green'>datatype:</font> type of the data
- <font color='green'>dest:</font> rank of the destination process within the communicator `comm`
- <font color='green'>tag:</font> nonnegative integer which is additional transferred with the message
  - Usage: the program can categorize the messages to identify one set to another. <p></p>
- Fortran binding:
```fortran
use MPI !or the older form: include 'mpif.h'
MPI_SEND(BUF, COUNT, DATATYPE, DEST, TAG, COMM,IERROR)
<type>    BUF(*)
INTEGER    COUNT, DATATYPE, DEST, TAG, COMM, IERROR
```
- Fortran 2008 binding:
```fortran
use MPI_F08
MPI_Send(buf, count, datatype, dest, tag, comm, ierror)
TYPE(*), DIMENSION(..), INTENT(IN) :: buf
INTEGER, INTENT(IN) :: count, dest, tag
TYPE(MPI_Datatype), INTENT(IN) :: datatype
TYPE(MPI_Comm), INTENT(IN) :: comm
INTEGER, OPTIONAL, INTENT(OUT) :: ierror
```

## MPI_RECV

- C/C++ binding:
```cpp
#include <mpi.h>
int MPI_Recv(void *buf, int count, MPI_Datatype datatype, int source,int tag, MPI_Comm comm, MPI_Status *status)
```
- <font color='green'>buf:</font> address of the first entry of the buffer in which the data will be stored
  - Must be large enough otherwise an <font color='red'>overflow error</font> occurs!
- <font color='green'>count:</font> The length of the received message must be less than or equal to the length of the receive buffer. The count argument indicates the maximum length of a message; the actual length of the message can be determined with `MPI_Get_count`.
- <font color='green'>source:</font> rank of the source (sender) process within the communicator `comm`
- <font color='green'>status:</font> contains information about the received message, to be explained!<p></p>
- Fortran binding:
```fortran
use MPI !or the older form: include 'mpif.h'
MPI_RECV(BUF,COUNT,DATATYPE, SOURCE,TAG,COMM,STATUS,IERROR)
<type>    BUF(*)
INTEGER   COUNT, DATATYPE, SOURCE, TAG, COMM
INTEGER   STATUS(MPI_STATUS_SIZE), IERROR
```
- Fortran 2008 binding:
```fortran
use MPI_F08
MPI_Send(buf, count, datatype, dest, tag, comm, ierror)
TYPE(*), DIMENSION(..), INTENT(IN) :: buf
INTEGER, INTENT(IN) :: count, dest, tag
TYPE(MPI_Datatype), INTENT(IN) :: datatype
TYPE(MPI_Comm), INTENT(IN) :: comm
INTEGER, OPTIONAL, INTENT(OUT) :: ierror
```

## Quiz

1. Which of the following is correct?
   <ol style="list-style-type: lower-alpha;">
   <li>There is a mechanism for automatic workload distribution in MPI</li>
   <li>MPI allows for data transfer through a communication network</li>
   <li>In MPI, workload can be split among processes according to their ranks</li>
   <li>To execute an application, the MPI standard prescribes a startup procedure</li>
   </ol>
   <details style="display: inline;">
       <summary style="display: inline; cursor: pointer;">
           <b>Answer: </b>
       </summary> b. and c.
   </details>
2. Is the rank of a process within a communicator unique?
   <ol style="list-style-type: lower-alpha;">
   <li>yes</li>
   <li>no</li>
   </ol>
   <details style="display: inline;">
       <summary style="display: inline; cursor: pointer;">
           <b>Answer: </b>
       </summary> a.
   </details>
3. Does <font color='green'>count</font> in `MPI_Send` and `MPI_Recv` determine the number of <font color='green'>bytes</font> in the point-to-point communication?
   <ol style="list-style-type: lower-alpha;">
   <li>yes</li>
   <li>no</li>
   </ol>
   <details style="display: inline;">
       <summary style="display: inline; cursor: pointer;">
           <b>Answer: </b>
       </summary> b.
   </details>


## Exercise 1: MPI “Hello World!” in C

```cpp
#include <mpi.h>

int main(char argc, char **argv) {
  int irank, nrank;

	MPI_FIXME(FIXME,FIXME);
	MPI_Comm_FIXME(MPI_COMM_WORLD, &nrank);
	MPI_Comm_rank(FIXME, FIXME);

	printf("Hello World! I am %d of %d\n", irank, nrak);

	MPI_FIXME();
}
```

<div class="callout note">
<strong>Note:</strong>
<span style="color:darkblue">
Replace all <code>FIXME</code> markers with the correct MPI expression!</span>
</div>

## Exercise 2: calculating $\pi$ using Monte Carlo method

In this exercise you practice:
- Workload distribution
- Eliminating repetition of work done by processes
- Collecting results of all processes

<font color='green'>Question:</font> Can we improve the accuracy by increasing the number random points, i.e. $nn>10^9$ ?